# Urban Heat & Cooling-Priority Mapping — Track B / UN1: U-Net Classifier

**NUS-ISS Practice Module, Week 3, Step 3 (deep classifier, after the RF baseline).**

Trains a U-Net semantic segmentation model on the same 4-class scheme
(vegetation / built_up / bare / water), same feature bands, same season
window, same validation-point exclusion as `train_rf_baseline.ipynb` (RF1) —
kept identical deliberately, since the whole point of comparing RF vs U-Net
is isolating the effect of model choice, not accidentally comparing
different inputs too.

**Why this pipeline looks different from RF1:** RF classifies pixel by
pixel, independently — each pixel only "sees" its own 9 feature values.
U-Net is a convolutional segmentation model — it needs spatial *patches* of
imagery as input (typically 128x128 pixels here), so it can use neighboring
pixels' context to inform each prediction. That's the actual hypothesis
being tested downstream: does spatial context help distinguish, e.g.,
built-up from bare (RF's weak point) better than per-pixel spectral values
alone?

**Pipeline shape:**
1. UN1.1-UN1.5 — same composite + label setup as RF1 (kept identical)
2. UN1.6 — export training data as TFRecord patches (Earth Engine's
   standard mechanism for getting imagery into TensorFlow)
3. UN1.7 — parse patches into a `tf.data` pipeline, with a sample-weight
   mask so patches partially outside Singapore's boundary don't corrupt
   the loss
4. UN1.8-UN1.9 — define and train a compact U-Net
5. UN1.10-UN1.11 — export inference patches over the FULL boundary, run
   the trained model, reconstruct a full classified raster from the
   predicted patches
6. UN1.12 — informal accuracy check (same caveat as RF1.9 — sanity check
   only, not the formal Step-4 evaluation)

**Compute note:** the TensorFlow training itself (UN1.9) runs in Colab's
own CPU/GPU, NOT against your Earth Engine quota — only the patch exports
(UN1.6, UN1.10) count against EECU-hours. If you're on the free Community
tier, consider switching to Contributor tier (1,000 EECU-hours/month, still
free) before running this, since patch export over all of Singapore is
more compute-intensive than RF1's simpler point-sampling.

**Known limitation, stated upfront rather than hidden:** inference
reconstruction uses non-overlapping patches, so there may be minor
prediction discontinuities at patch boundaries ("tile seams"). Acceptable
for a baseline comparison; worth mentioning in your methods write-up.

Run top to bottom. Training (UN1.9) is the slow step — expect it to take a
while depending on Colab's assigned hardware (GPU strongly recommended;
check Runtime → Change runtime type).

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [4]:
# --- SETUP CELL 1: Install dependencies -------------------------------------
!pip install -q earthengine-api geemap pandas numpy rasterio tensorflow pyproj


## Setup 2 — Authenticate & initialize Earth Engine

In [5]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- must match SB1 / RF1 / Track A's project

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


## Setup 3 — Mount Google Drive

In [6]:
# --- SETUP CELL 3: Mount Google Drive ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 4 — Seeding + shared results tracker

In [7]:
# --- SETUP CELL 4: Seeding + shared results tracker --------------------------
import random
import numpy as np
import tensorflow as tf

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

track_b_results = {}
print("Seeds set, track_b_results initialized.")
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)
if len(tf.config.list_physical_devices('GPU')) == 0:
    print("⚠️  No GPU detected — training will be slow. Runtime → Change runtime type → GPU.")


Seeds set, track_b_results initialized.
GPU available: True


---
# UN1 — U-Net (train + classify)


## UN1.1 — Config (identical to RF1's config where it matters for comparability)

In [8]:
# --- UN1 CELL 1: Config -------------------------------------------------------
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

# Locked C4 season window — IDENTICAL to SB1 / RF1 / Track A's notebooks.
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
DRY_SEASON_MONTHS = [4, 5, 10, 11]
S2_CLOUD_PROB_MAX = 70

TARGET_SCALE = 10
S2_UTM_CRS = "EPSG:32648"

WORLDCOVER_ASSET = "ESA/WorldCover/v200/2021"
SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"

# 4-class bucket scheme — IDENTICAL to SB1 / RF1
WC_TREE, WC_SHRUB, WC_GRASS, WC_CROP = 10, 20, 30, 40
WC_BUILTUP, WC_BARE, WC_SNOWICE, WC_WATER = 50, 60, 70, 80
WC_WETLAND, WC_MANGROVE, WC_MOSSLICHEN = 90, 95, 100

BUCKET_VEGETATION, BUCKET_BUILTUP, BUCKET_BARE, BUCKET_WATER = 1, 2, 3, 4
BUCKET_NAMES = {
    BUCKET_VEGETATION: "vegetation", BUCKET_BUILTUP: "built_up",
    BUCKET_BARE: "bare", BUCKET_WATER: "water",
}
WC_TO_BUCKET_FROM = [WC_TREE, WC_SHRUB, WC_GRASS, WC_CROP, WC_BUILTUP,
                      WC_BARE, WC_SNOWICE, WC_WATER, WC_WETLAND,
                      WC_MANGROVE, WC_MOSSLICHEN]
WC_TO_BUCKET_TO = [BUCKET_VEGETATION, BUCKET_VEGETATION, BUCKET_VEGETATION,
                    BUCKET_VEGETATION, BUCKET_BUILTUP, BUCKET_BARE, 0,
                    BUCKET_WATER, BUCKET_VEGETATION, BUCKET_VEGETATION,
                    BUCKET_VEGETATION]

# Feature bands — IDENTICAL to RF1, so both models see the same inputs.
S2_FEATURE_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
INDEX_BANDS = ["NDVI", "NDBI", "NDWI"]
ALL_FEATURE_BANDS = S2_FEATURE_BANDS + INDEX_BANDS
N_FEATURE_BANDS = len(ALL_FEATURE_BANDS)
N_CLASSES = 4

# Validation exclusion — IDENTICAL to RF1
VALIDATION_EXCLUSION_BUFFER_M = 15

# U-Net specific config
PATCH_SIZE = 128          # pixels per side (128 * 10m = 1.28km per patch)
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
UNET_BASE_FILTERS = 32
EARLY_STOP_PATIENCE = 5
TRAIN_VAL_SPLIT = 0.85    # patch-level split, not pixel-level

DRIVE_DIR = "/content/drive/MyDrive/urban_heat_sg"
VALIDATION_CSV = f"{DRIVE_DIR}/validation_sample_200_labeled.csv"

TRAIN_PATCH_FOLDER = "unet_train_patches"
TRAIN_PATCH_PREFIX = "unet_train"
INFERENCE_PATCH_FOLDER = "unet_inference_patches"
INFERENCE_PATCH_PREFIX = "unet_inference"
MODEL_SAVE_PATH = f"{DRIVE_DIR}/unet_landcover_model.keras"
CLASSIFIED_RASTER_PATH = f"{DRIVE_DIR}/unet_landcover_classified.tif"

print(f"Feature bands ({N_FEATURE_BANDS}): {ALL_FEATURE_BANDS}")
print(f"Patch size: {PATCH_SIZE}x{PATCH_SIZE}, batch: {BATCH_SIZE}, epochs: {EPOCHS}")


Feature bands (9): ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']
Patch size: 128x128, batch: 8, epochs: 30


## UN1.2 — Load Singapore boundary (cached locally, falls back to API)

Checks for a cached `ura_subzones.geojson` in Drive first — skips the
data.gov.sg API call entirely if it's already there. Saves a copy to the
cache folder on first run so subsequent runs (and SB1/RF1, if updated the
same way) don't need the network round-trip either.


In [9]:
# --- UN1 CELL 2: Load Singapore boundary (cached) ------------------------------
import requests
import json as _json
import os

CACHE_DIR = "/content/drive/MyDrive/urban_heat_sg/cache"
SUBZONE_CACHE_PATH = f"{CACHE_DIR}/ura_subzones.geojson"

os.makedirs(CACHE_DIR, exist_ok=True)

if os.path.exists(SUBZONE_CACHE_PATH):
    print(f"Loading cached subzones from {SUBZONE_CACHE_PATH} — skipping API call.")
    with open(SUBZONE_CACHE_PATH) as f:
        subzone_geojson = _json.load(f)
else:
    print("No cache found — fetching from data.gov.sg (one-time; will cache for next time).")

    def fetch_datagovsg_geojson(dataset_id):
        poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
        r = requests.get(poll_url)
        r.raise_for_status()
        payload = r.json()
        if payload.get("code") != 0:
            raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
        return requests.get(payload["data"]["url"]).json()

    subzone_geojson = fetch_datagovsg_geojson(SUBZONE_DATASET_ID)

    with open(SUBZONE_CACHE_PATH, "w") as f:
        _json.dump(subzone_geojson, f)
    print(f"Cached to {SUBZONE_CACHE_PATH} for future runs (SB1/RF1 too, if pointed at the same cache).")

for feature in subzone_geojson.get("features", []):
    props = feature.get("properties", {})
    for old_key in list(props.keys()):
        if "." in old_key:
            props[old_key.replace(".", "_")] = props.pop(old_key)

subzones = ee.FeatureCollection(subzone_geojson)
sg_boundary = subzones.union(1).first().geometry()
print(f"Singapore boundary built from {subzones.size().getInfo()} subzones.")


Loading cached subzones from /content/drive/MyDrive/urban_heat_sg/cache/ura_subzones.geojson — skipping API call.
Singapore boundary built from 332 subzones.


## UN1.3 — Cloud masking + season-filter helpers (same as RF1.3)

In [10]:
# --- UN1 CELL 3: Cloud masking + season-filter helpers -----------------------
def mask_s2_clouds(cloud_prob_image):
    return cloud_prob_image.select("probability").lt(S2_CLOUD_PROB_MAX)


def date_filter_for_years_months(collection, years, months):
    filters = []
    for y in years:
        for m in months:
            start = ee.Date.fromYMD(y, m, 1)
            end = start.advance(1, "month")
            filters.append(ee.Filter.date(start, end))
    return collection.filter(ee.Filter.Or(*filters))


print("Helpers defined.")


Helpers defined.


## UN1.4 — Build Sentinel-2 composite + indices (IDENTICAL to RF1.4)

Same code as RF1.4, deliberately — both classifiers must see the same input
pixel values for the comparison to isolate model choice, not input drift.


In [11]:
# --- UN1 CELL 4: Sentinel-2 composite + indices -------------------------------
s2_sr = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(sg_bbox)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
)
s2_sr = date_filter_for_years_months(s2_sr, YEARS, DRY_SEASON_MONTHS)
print("Sentinel-2 scenes after season filter (pre-mask):", s2_sr.size().getInfo())

s2_cloud_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(sg_bbox)
s2_cloud_prob = date_filter_for_years_months(s2_cloud_prob, YEARS, DRY_SEASON_MONTHS)

joined = ee.Join.saveFirst("cloud_mask").apply(
    primary=s2_sr, secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
)

def _mask(img):
    img = ee.Image(img)
    cloud_img = ee.Image(img.get("cloud_mask"))
    return img.updateMask(mask_s2_clouds(cloud_img))

s2_masked = ee.ImageCollection(joined).map(_mask)
print("Sentinel-2 usable scenes (post-mask):", s2_masked.size().getInfo())

composite_bands = s2_masked.select(S2_FEATURE_BANDS).median().clip(sg_boundary)
composite_bands = composite_bands.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE)

ndvi = composite_bands.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndbi = composite_bands.normalizedDifference(["B11", "B8"]).rename("NDBI")
ndwi = composite_bands.normalizedDifference(["B3", "B8"]).rename("NDWI")

feature_image = ee.Image.cat([composite_bands, ndvi, ndbi, ndwi]).select(ALL_FEATURE_BANDS)
valid_mask = feature_image.select("B4").mask()

proj_check = feature_image.projection().getInfo()
print(f"Feature image projection: {proj_check['crs']} (expect {S2_UTM_CRS})")


Sentinel-2 scenes after season filter (pre-mask): 69
Sentinel-2 usable scenes (post-mask): 69
Feature image projection: EPSG:32648 (expect EPSG:32648)


## UN1.5 — WorldCover training labels (4-class) + validation-point exclusion

Same non-circularity enforcement as RF1.5 — training patches are drawn from
a region that explicitly excludes a buffer around every validation point,
not just assumed to.


In [12]:
# --- UN1 CELL 5: Training labels + validation exclusion -----------------------
import pandas as pd

worldcover_raw = ee.Image(WORLDCOVER_ASSET).select("Map")
worldcover = worldcover_raw.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE).clip(sg_boundary)

wc_bucket = worldcover.remap(WC_TO_BUCKET_FROM, WC_TO_BUCKET_TO, 0).rename("wc_class")
wc_bucket = wc_bucket.updateMask(wc_bucket.neq(0)).updateMask(valid_mask)

val_df = pd.read_csv(VALIDATION_CSV)
print(f"Loaded {len(val_df)} validation points from {VALIDATION_CSV}")

val_geoms = [ee.Geometry.Point([row.lon, row.lat]) for row in val_df.itertuples()]
val_points_fc = ee.FeatureCollection([ee.Feature(g) for g in val_geoms])
val_buffer = val_points_fc.geometry().buffer(VALIDATION_EXCLUSION_BUFFER_M)

training_region = sg_boundary.difference(val_buffer, ee.ErrorMargin(1))

sg_area = sg_boundary.area(1).getInfo()
training_area = training_region.area(1).getInfo()
excluded_area = sg_area - training_area
print(f"Training region area (post-exclusion): {training_area/1e6:,.2f} km² "
      f"(excluded {excluded_area:,.0f} m² around validation points)")
if excluded_area < 1000:
    print("⚠️  Exclusion area suspiciously small — check val_df before trusting this.")
else:
    print("✅ Validation points are spatially excluded from the training region.")

# wc_class values are 1-4 (buckets); for the label band, fill unmasked-out
# pixels with 0 (sentinel meaning "no valid label") so the export always has
# a defined value per pixel — the sample-weight mask in UN1.7 uses this to
# exclude those pixels from the loss.
training_stack = ee.Image.cat([feature_image, wc_bucket.unmask(0)]).clip(training_region)
print("Training stack bands:", training_stack.bandNames().getInfo())


Loaded 200 validation points from /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeled.csv
Training region area (post-exclusion): 788.16 km² (excluded 139,654 m² around validation points)
✅ Validation points are spatially excluded from the training region.
Training stack bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI', 'wc_class']


## UN1.6 — Export training patches as TFRecord

Earth Engine's standard mechanism for getting imagery into TensorFlow: tiles
the training region into fixed-size patches and writes them as `.tfrecord.gz`
files, plus a `mixer.json` describing the patch grid layout. Async export —
this cell submits the task and polls until done (can take a while for a
full-Singapore-sized region at 128x128 patches).


In [13]:
# --- UN1 CELL 6: Export training patches ---------------------------------------
import time

train_export_task = ee.batch.Export.image.toDrive(
    image=training_stack,
    description=TRAIN_PATCH_PREFIX,
    folder=TRAIN_PATCH_FOLDER,
    fileNamePrefix=TRAIN_PATCH_PREFIX,
    region=training_region,
    scale=TARGET_SCALE,
    crs=S2_UTM_CRS,
    maxPixels=1e13,
    fileFormat="TFRecord",
    formatOptions={"patchDimensions": [PATCH_SIZE, PATCH_SIZE], "compressed": True},
)
train_export_task.start()
print(f"Training patch export started: {TRAIN_PATCH_PREFIX}")

while train_export_task.active():
    print(f"  ...{train_export_task.status()['state']}")
    time.sleep(20)

train_export_status = train_export_task.status()
print(f"\nFinal status: {train_export_status['state']}")
if train_export_status["state"] != "COMPLETED":
    raise RuntimeError(f"Training patch export did not complete: {train_export_status}")
print(f"✅ Patches written to Drive folder: {TRAIN_PATCH_FOLDER}")
print("Look for .tfrecord.gz files and a mixer.json in that folder before continuing.")


Training patch export started: unet_train
  ...READY
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING

Final status: COMPLETED
✅ Patches written to Drive folder: unet_train_patches
Look for .tfrecord.gz files and a mixer.json in that folder before continuing.


## UN1.7 — Parse patches into a tf.data pipeline

Each band is a separate feature in the TFRecord, shape `[PATCH_SIZE,
PATCH_SIZE]`. The label band (`wc_class`) uses 0 as "no valid label" — a
per-pixel sample-weight mask excludes those pixels from the loss so patches
that straddle the training-region boundary (irregular shape, since it's
Singapore's real boundary minus validation-point circles) don't corrupt
training with fake labels.


In [14]:
# --- UN1 CELL 7: Parse TFRecord patches -----------------------------------------
import glob

train_tfrecord_files = sorted(glob.glob(f"/content/drive/MyDrive/{TRAIN_PATCH_FOLDER}/{TRAIN_PATCH_PREFIX}*.tfrecord.gz"))
print(f"Found {len(train_tfrecord_files)} training TFRecord file(s).")
if len(train_tfrecord_files) == 0:
    raise FileNotFoundError(f"No TFRecord files found in {TRAIN_PATCH_FOLDER} — check UN1.6 completed.")

ALL_BANDS = ALL_FEATURE_BANDS + ["wc_class"]
feature_description = {
    band: tf.io.FixedLenFeature(shape=[PATCH_SIZE, PATCH_SIZE], dtype=tf.float32)
    for band in ALL_BANDS
}

def parse_tfrecord(example_proto):
    parsed = tf.io.parse_single_example(example_proto, feature_description)
    features = tf.stack([parsed[b] for b in ALL_FEATURE_BANDS], axis=-1)  # (H, W, N_FEATURE_BANDS)
    label_raw = parsed["wc_class"]  # (H, W), values 0 (nodata) or 1-4 (bucket)

    weight = tf.cast(label_raw > 0, tf.float32)  # 0 where nodata, 1 where valid
    label_idx = tf.cast(tf.maximum(label_raw - 1, 0), tf.int32)  # shift 1-4 -> 0-3; nodata->0 (masked by weight anyway)
    label_onehot = tf.one_hot(label_idx, depth=N_CLASSES)

    return features, label_onehot, weight


raw_dataset = tf.data.TFRecordDataset(train_tfrecord_files, compression_type="GZIP")
parsed_dataset = raw_dataset.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)

# Drop patches that are entirely nodata (weight sums to 0) — wasted training signal.
def has_valid_pixels(features, label, weight):
    return tf.reduce_sum(weight) > 0

parsed_dataset = parsed_dataset.filter(has_valid_pixels)

all_patches = list(parsed_dataset.as_numpy_iterator())
n_patches = len(all_patches)
print(f"Usable patches (after dropping all-nodata ones): {n_patches}")

rng = np.random.default_rng(RANDOM_SEED)
indices = rng.permutation(n_patches)
split_idx = int(n_patches * TRAIN_VAL_SPLIT)
train_indices, val_indices = indices[:split_idx], indices[split_idx:]
print(f"Train patches: {len(train_indices)}, validation patches: {len(val_indices)}")

def make_dataset(patch_list, indices, batch_size, shuffle):
    features = np.stack([patch_list[i][0] for i in indices])
    labels = np.stack([patch_list[i][1] for i in indices])
    weights = np.stack([patch_list[i][2] for i in indices])
    ds = tf.data.Dataset.from_tensor_slices((features, labels, weights))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(indices), seed=RANDOM_SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(all_patches, train_indices, BATCH_SIZE, shuffle=True)
val_ds = make_dataset(all_patches, val_indices, BATCH_SIZE, shuffle=False)
print("tf.data pipelines built.")


Found 1 training TFRecord file(s).


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_3_device_/job:localhost/replica:0/task:0/device:CPU:0}} Error in user-defined function passed to ParallelMapDatasetV2:2 transformation with iterator: Iterator::Root::Prefetch::Filter::ParallelMapV2: Key: wc_class.  Data types don't match. Data type: string but expected type: float
	 [[{{node ParseSingleExample/ParseExample/ParseExampleV2}}]] [Op:IteratorGetNext] name: 

## UN1.8 — Define a compact U-Net

In [ ]:
# --- UN1 CELL 8: Define U-Net ----------------------------------------------------
from tensorflow.keras import layers, models

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

def build_unet(input_shape, n_classes, base_filters=UNET_BASE_FILTERS):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1 = conv_block(inputs, base_filters)
    p1 = layers.MaxPooling2D(2)(c1)
    c2 = conv_block(p1, base_filters * 2)
    p2 = layers.MaxPooling2D(2)(c2)
    c3 = conv_block(p2, base_filters * 4)
    p3 = layers.MaxPooling2D(2)(c3)

    # Bottleneck
    b = conv_block(p3, base_filters * 8)

    # Decoder
    u3 = layers.Conv2DTranspose(base_filters * 4, 2, strides=2, padding="same")(b)
    u3 = layers.Concatenate()([u3, c3])
    d3 = conv_block(u3, base_filters * 4)

    u2 = layers.Conv2DTranspose(base_filters * 2, 2, strides=2, padding="same")(d3)
    u2 = layers.Concatenate()([u2, c2])
    d2 = conv_block(u2, base_filters * 2)

    u1 = layers.Conv2DTranspose(base_filters, 2, strides=2, padding="same")(d2)
    u1 = layers.Concatenate()([u1, c1])
    d1 = conv_block(u1, base_filters)

    outputs = layers.Conv2D(n_classes, 1, activation="softmax")(d1)
    return models.Model(inputs, outputs, name="unet_landcover")


unet_model = build_unet((PATCH_SIZE, PATCH_SIZE, N_FEATURE_BANDS), N_CLASSES)
unet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    weighted_metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
)
unet_model.summary()


## UN1.9 — Train

In [ ]:
# --- UN1 CELL 9: Train ------------------------------------------------------------
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=EARLY_STOP_PATIENCE, restore_best_weights=True,
)

# sample_weight is passed as the 3rd element of each dataset tuple —
# tf.keras.Model.fit handles this automatically for (features, labels, weight) datasets.
history = unet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
)

unet_model.save(MODEL_SAVE_PATH)
print(f"\nModel saved to {MODEL_SAVE_PATH}")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")


## UN1.10 — Export inference patches (FULL Singapore boundary, features only)

Unlike training, inference needs predictions everywhere, including at/near
validation points — so this exports over the full `sg_boundary`, not the
training-excluded region.


In [ ]:
# --- UN1 CELL 10: Export inference patches --------------------------------------
inference_stack = feature_image.clip(sg_boundary)

inference_export_task = ee.batch.Export.image.toDrive(
    image=inference_stack,
    description=INFERENCE_PATCH_PREFIX,
    folder=INFERENCE_PATCH_FOLDER,
    fileNamePrefix=INFERENCE_PATCH_PREFIX,
    region=sg_boundary,
    scale=TARGET_SCALE,
    crs=S2_UTM_CRS,
    maxPixels=1e13,
    fileFormat="TFRecord",
    formatOptions={"patchDimensions": [PATCH_SIZE, PATCH_SIZE], "compressed": True},
)
inference_export_task.start()
print(f"Inference patch export started: {INFERENCE_PATCH_PREFIX}")

while inference_export_task.active():
    print(f"  ...{inference_export_task.status()['state']}")
    time.sleep(20)

inference_export_status = inference_export_task.status()
print(f"\nFinal status: {inference_export_status['state']}")
if inference_export_status["state"] != "COMPLETED":
    raise RuntimeError(f"Inference patch export did not complete: {inference_export_status}")
print(f"✅ Inference patches written to Drive folder: {INFERENCE_PATCH_FOLDER}")


## UN1.11 — Run inference + reconstruct full raster

Reads `mixer.json` (patch grid layout + georeferencing, written automatically
alongside the patches) to know how to lay predicted patches back into a
single georeferenced raster.


In [ ]:
# --- UN1 CELL 11: Inference + reconstruction --------------------------------------
import rasterio
from rasterio.transform import Affine

inference_dir = f"/content/drive/MyDrive/{INFERENCE_PATCH_FOLDER}"
inference_tfrecord_files = sorted(glob.glob(f"{inference_dir}/{INFERENCE_PATCH_PREFIX}*.tfrecord.gz"))
mixer_path = f"{inference_dir}/{INFERENCE_PATCH_PREFIX}mixer.json"

print(f"Found {len(inference_tfrecord_files)} inference TFRecord file(s).")
with open(mixer_path) as f:
    mixer = _json.load(f)
print("Mixer metadata:", {k: mixer[k] for k in mixer if k != "projection"})

patches_per_row = mixer["patchesPerRow"]
patches_per_col = mixer["totalPatches"] // patches_per_row
proj = mixer["projection"]
affine_params = proj["affine"]["doubleMatrix"] if "doubleMatrix" in proj.get("affine", {}) else None

# Feature description for inference patches (no label band this time)
inference_feature_description = {
    band: tf.io.FixedLenFeature(shape=[PATCH_SIZE, PATCH_SIZE], dtype=tf.float32)
    for band in ALL_FEATURE_BANDS
}

def parse_inference_tfrecord(example_proto):
    parsed = tf.io.parse_single_example(example_proto, inference_feature_description)
    return tf.stack([parsed[b] for b in ALL_FEATURE_BANDS], axis=-1)

inference_raw = tf.data.TFRecordDataset(inference_tfrecord_files, compression_type="GZIP")
inference_parsed = inference_raw.map(parse_inference_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
inference_patches = list(inference_parsed.as_numpy_iterator())
print(f"Loaded {len(inference_patches)} inference patches for prediction.")

# Predict class per pixel for every patch, in original patch order (EE writes
# patches in row-major order matching mixer.json's patchesPerRow).
full_raster = np.zeros((patches_per_col * PATCH_SIZE, patches_per_row * PATCH_SIZE), dtype=np.uint8)

for idx, patch in enumerate(inference_patches):
    pred = unet_model.predict(patch[np.newaxis, ...], verbose=0)[0]  # (H, W, N_CLASSES)
    pred_class = np.argmax(pred, axis=-1).astype(np.uint8) + 1  # shift 0-3 back to bucket ids 1-4
    row = idx // patches_per_row
    col = idx % patches_per_row
    full_raster[row*PATCH_SIZE:(row+1)*PATCH_SIZE, col*PATCH_SIZE:(col+1)*PATCH_SIZE] = pred_class
    if (idx + 1) % 50 == 0:
        print(f"  ...predicted {idx+1}/{len(inference_patches)} patches")

print("Prediction complete. Reconstructing GeoTIFF...")

# Build the affine transform from mixer.json's projection info.
crs_str = proj["crs"]
if affine_params:
    transform = Affine(affine_params[0], affine_params[1], affine_params[2],
                        affine_params[3], affine_params[4], affine_params[5])
else:
    # Fallback: derive from scale + region bounds if doubleMatrix isn't present —
    # verify this against a known landmark before trusting it fully.
    bounds = sg_boundary.bounds().getInfo()["coordinates"][0]
    minx = min(p[0] for p in bounds)
    maxy = max(p[1] for p in bounds)
    transform = Affine(TARGET_SCALE, 0, minx, 0, -TARGET_SCALE, maxy)
    print("⚠️  Using fallback affine transform — verify raster alignment before trusting it.")

with rasterio.open(
    CLASSIFIED_RASTER_PATH, "w", driver="GTiff",
    height=full_raster.shape[0], width=full_raster.shape[1],
    count=1, dtype=full_raster.dtype, crs=crs_str, transform=transform,
) as dst:
    dst.write(full_raster, 1)

print(f"✅ Classified raster written to {CLASSIFIED_RASTER_PATH}")


## UN1.12 — Informal accuracy check (NOT the formal Step-4 evaluation)

Same caveat as RF1.9 — sanity check only. The real confusion matrix /
per-class F1 / comparison table happens in the shared evaluation notebook
once RF, U-Net, and the ensemble are all scored identically.


In [ ]:
# --- UN1 CELL 12: Informal accuracy check ---------------------------------------
with rasterio.open(CLASSIFIED_RASTER_PATH) as src:
    raster_data = src.read(1)
    raster_transform = src.transform

val_df["agreed_label"] = val_df["agreed_label"].fillna("").astype(str)
valid_rows = val_df[val_df["agreed_label"].isin(BUCKET_NAMES.values())].copy()
n_skipped = len(val_df) - len(valid_rows)
if n_skipped > 0:
    print(f"Skipping {n_skipped} validation point(s) with blank/uncertain labels.")

def lonlat_to_pixel(lon, lat, transform):
    col, row = ~transform * (lon, lat)
    return int(row), int(col)

# Points need to be reprojected from lon/lat (EPSG:4326) to the raster's CRS
# (EPSG:32648) before converting to pixel coordinates.
from pyproj import Transformer
transformer = Transformer.from_crs("EPSG:4326", crs_str, always_xy=True)

preds = []
for row in valid_rows.itertuples():
    x, y = transformer.transform(row.lon, row.lat)
    r, c = lonlat_to_pixel(x, y, raster_transform)
    if 0 <= r < raster_data.shape[0] and 0 <= c < raster_data.shape[1]:
        preds.append(int(raster_data[r, c]))
    else:
        preds.append(None)

valid_rows["unet_pred_bucket"] = preds
valid_rows["unet_pred_name"] = valid_rows["unet_pred_bucket"].map(BUCKET_NAMES)
BUCKET_NAME_TO_ID = {v: k for k, v in BUCKET_NAMES.items()}
valid_rows["true_bucket"] = valid_rows["agreed_label"].map(BUCKET_NAME_TO_ID)

scored = valid_rows.dropna(subset=["unet_pred_bucket"])
accuracy = (scored["unet_pred_bucket"] == scored["true_bucket"]).mean()
print(f"\nInformal U-Net accuracy on {len(scored)} validation points: {accuracy*100:.1f}%")
print("(Sanity check only — run the formal evaluation notebook for the real deliverable numbers.)")

print("\nQuick cross-tab (rows=true, columns=predicted):")
print(pd.crosstab(scored["agreed_label"], scored["unet_pred_name"]))


## UN1.13 — Verdict

In [ ]:
# --- UN1 CELL 13: Verdict ----------------------------------------------------------
print("\n--- UN1 Verdict ---")

un1_checks = {
    "Training patches exported": train_export_status["state"] == "COMPLETED",
    "Usable training patches > 0": n_patches > 0,
    "Model trained (history recorded)": len(history.history["loss"]) > 0,
    "Inference patches exported": inference_export_status["state"] == "COMPLETED",
    "Full raster reconstructed": full_raster.sum() > 0,
    "Informal accuracy computed": len(scored) > 0,
}

for check, passed in un1_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

un1_pass = all(un1_checks.values())
print(f"\n{'✅ UN1 PASS' if un1_pass else '⚠️  UN1 FAIL — review flagged checks'}")

track_b_results["UN1_unet"] = {
    "status": "PASS" if un1_pass else "FAIL",
    "checks": un1_checks,
    "n_training_patches": n_patches,
    "epochs_trained": len(history.history["loss"]),
    "best_val_loss": float(min(history.history["val_loss"])),
    "informal_accuracy": float(accuracy),
    "hyperparams": {
        "patch_size": PATCH_SIZE, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE, "base_filters": UNET_BASE_FILTERS,
        "seed": RANDOM_SEED,
    },
}
